# UFTC B-747 — coordinated banked turn with mid-turn engine failure

This notebook is the **maneuver** counterpart of the stationary-trim Phase 1 / Phase 3 / Phase 4 demos. Instead of holding straight-and-level cruise, the aircraft executes a 300 s scenario: a coordinated banked-turn maneuver followed by an extended new-heading hold:

| Window (s) | Phase | What changes |
|------------|-------|--------------|
| 0 – 5      | level cruise | hold V, h, ψ = 0 |
| 5 – 15     | roll-in      | bank φ ramps 0 → +15° (right bank) |
| 15 – 127.95 | steady banked turn | φ = +15°, ψ slews at ≈0.73°/s (consistent with `g·tan(φ)/V`) |
| 127.95 – 137.95 | roll-out | φ ramps +15° → 0 |
| 137.95 – 300 | new-heading hold | φ = 0, ψ = ψ_final |

**Damage**: the **left-outer engine flames out at t = 50 s**, *mid-turn*, so UFTC must keep tracking a slewing reference under asymmetric thrust.

We compare two configurations:
* **Phase 1** — L2 AA-INDI inner + L3 IADP middle + envelope allocator, **no L4 outer, no monitor**.
* **Phase 4** — full UFTC stack: pre-trained L4 D-SAC outer + composite Lyapunov monitor + macro-action dispatcher.

**Plant-agnostic note** — UFTC's reference vector remains `0` throughout. The time-varying maneuver is injected in the *envelope-allocator setpoint* `ref0(t)` that `uftc_state_transform` uses to build the error-form state. UFTC sees a non-trivial drifting error and the cascade compensates online.


## 1. Imports and simulation settings


In [ ]:
import warnings                                            # standard-library warnings module
warnings.filterwarnings("ignore")                            # silence noisy NumPy/Torch warnings during the demo

import math                                                  # scalar math (radians, tan, sqrt, ...)
from pathlib import Path                                     # cross-platform path manipulation
from collections import Counter                              # used to tally macro-action kinds at the end

import matplotlib.pyplot as plt                              # plotting backend for time-history figures
import numpy as np                                           # numerical arrays / linear algebra
from scipy.optimize import fsolve                            # nonlinear-equation solver for engine-out trim

from tensoraerospace.aerospacemodel.b747.nonlinear import B747Configuration, trim   # nominal config + trim solver
from tensoraerospace.aerospacemodel.b747.nonlinear.damage.state import B747DamageState  # full damage-state vector
from tensoraerospace.aerospacemodel.b747.nonlinear.dynamics import b747_ode_6dof    # 6-DoF ODE used in trim residuals
from tensoraerospace.aerospacemodel.b747.nonlinear.damage import (
    DamageProfile, EngineFailureEvent,                       # programmable damage timeline + engine event
)
from tensoraerospace.aerospacemodel.b747.nonlinear.engine import JT9DEngine          # JT9D thrust model (per cluster)
from tensoraerospace.aerospacemodel.b747.nonlinear.params import (
    default_parameters, isa_speed_of_sound_ft_s,             # nominal parameters + ISA Mach helper
)
import importlib                                             # used below to reload the controller module
import tensoraerospace.agent.uftc.controller as uftc_controller_module  # UFTC top-level orchestrator module
importlib.reload(uftc_controller_module)                     # pick up local edits without restarting the kernel
from tensoraerospace.agent.uftc.controller import UFTCConfig, UFTCController  # config dataclass + controller class
from tensoraerospace.agent.aa_indi.model import AAINDIConfig                  # L2 inner-loop config
from tensoraerospace.agent.iadp.model import IADPConfig                       # L3 middle-loop config
from tensoraerospace.agent.uftc.fdd.detector import FDDConfig                 # FDD/Kalman detector config
from tensoraerospace.envs.b747_nonlinear import NonlinearB747Env              # gym-like nonlinear B-747 environment

np.random.seed(0)                                            # determinism for NumPy RNG
import torch                                                 # PyTorch (used inside the L4 D-SAC actor/critic)
torch.manual_seed(0)                                         # determinism for Torch RNG (CPU + CUDA)

DT = 0.05                                                    # simulation step in seconds (20 Hz)
TOTAL_TIME = 300.0                                           # total episode horizon in seconds
DAMAGE_TIME = 50.0                                           # wall-clock time at which the engine flames out
N_EP = int(TOTAL_TIME / DT)         # 6000 steps             # number of integration steps in one episode
DAMAGE_STEP = int(DAMAGE_TIME / DT)                          # step index at which the failure is triggered

V_REF_FT_S = 674.0                                           # cruise true airspeed (ft/s) ≈ M0.7 at FL200
ALT_REF_FT = 20_000.0                                        # cruise altitude (feet)
PSI_REF_DEG = 0.0                                            # initial heading (deg) — north

# Coordinated-turn schedule. Sign convention: positive bank phi = right wing
# down (right bank). For a coordinated right turn the heading rate satisfies
#   psi_dot = (g/V) * tan(phi),  positive bank -> positive psi rate.
TURN_BANK_DEG  = 15.0     # peak bank angle (right turn -> positive bank)
TURN_BANK_SIGN = +1.0     # right turn
ROLL_IN_S   = (5.0, 15.0)                                    # window during which phi ramps 0 -> +15 deg
STEADY_S    = (15.0, 127.95)                                 # steady-bank window at peak phi
ROLL_OUT_S  = (127.95, 137.95)                               # window during which phi ramps back to 0

print(f"Episode horizon : {TOTAL_TIME:.0f} s ({N_EP} steps @ dt={DT}s)")     # sanity-print of horizon
print(f"Damage trigger  : t = {DAMAGE_TIME:.0f} s (step {DAMAGE_STEP})")     # sanity-print of failure step
print(f"Turn schedule   : roll-in {ROLL_IN_S}, steady {STEADY_S}, roll-out {ROLL_OUT_S}")  # turn windows
print(f"Peak bank       : {TURN_BANK_SIGN*TURN_BANK_DEG:+.1f} deg (right bank)")           # signed peak bank


## 2. Cruise and one-engine-out trim

Same trim solve as Phase 4: a healthy cruise trim plus a static left-outer-engine-out trim that the envelope allocator blends into during the failure transient. The blend coefficient is `engine_loss_estimate ∈ [0, 1]` driven by the FDD severity output.


In [ ]:
trim_result = trim(altitude_ft=ALT_REF_FT, V_ft_s=V_REF_FT_S,    # solve healthy cruise trim at FL200
                   config=B747Configuration.NOMINAL)                # nominal aerodynamic / engine config
assert trim_result.converged                                        # fail fast if trim solver did not converge

delta_e_trim_rad = float(trim_result.elevator_rad)                  # elevator angle that holds 1g cruise
throttle_trim = float(trim_result.throttle)                         # throttle setting (in [0,1]) for cruise
theta_ref_deg = math.degrees(trim_result.theta_rad)                 # pitch attitude in degrees, for printout
healthy_trim_action = np.array([                                    # healthy 4-channel trim action vector
    delta_e_trim_rad, 0.0, 0.0, throttle_trim,                      # de, da=0, dr=0, throttle
], dtype=np.float64)

engine_params = default_parameters(B747Configuration.NOMINAL)       # nominal parameter struct (mass, inertia, ...)
engine_model = JT9DEngine(                                          # JT9D thrust model used to back out per-engine thrust
    n_engines=4,                                                    # B-747 has four engines
    sls_thrust_per_engine_lb=engine_params.engine_thrust_max_lb / 4.0,  # SLS thrust split equally
    idle_frac=engine_params.engine_idle_frac,                       # minimum idle thrust fraction
    spool_tau_s=engine_params.engine_tau_s,                         # spool-up time constant (seconds)
)

engine_out_params = default_parameters(B747Configuration.NOMINAL)   # parameter copy used for engine-out trim
engine_out_damage_state = B747DamageState.healthy()                 # start from a healthy damage state
engine_out_damage_state.engines_mu[1] = 0.0                         # zero the multiplicative mu of engine #1 (left outer)
engine_out_params.damage_state = engine_out_damage_state            # attach the engine-out damage state
throttle_engine_out_estimate = min(throttle_trim * 4.0 / 3.0, 1.0)  # rough initial guess: redistribute thrust over 3 engines

def engine_out_state_from_vars(alpha_rad, beta_rad, theta_rad):     # build the 12-state body-axis state vector
    return np.array([                                               # used inside fsolve as a closure variable
        V_REF_FT_S * math.cos(alpha_rad) * math.cos(beta_rad),      # u — body x-axis velocity
        V_REF_FT_S * math.sin(beta_rad),                            # v — body y-axis velocity (sideslip)
        V_REF_FT_S * math.sin(alpha_rad) * math.cos(beta_rad),      # w — body z-axis velocity
        0.0, 0.0, 0.0,                                              # p, q, r — body angular rates (zeroed in trim)
        0.0, theta_rad, 0.0,                                        # phi=0, theta, psi=0 (level except for AoA)
        0.0, 0.0, -ALT_REF_FT,                                      # x_e=0, y_e=0, z_e=-altitude (NED)
    ], dtype=np.float64)

def engine_out_trim_residual(z):                                    # residual whose root is the engine-out trim
    alpha_rad, beta_rad, theta_rad, de_rad, da_rad, dr_rad, throttle_cmd = z  # unpack 7 free variables
    x = engine_out_state_from_vars(alpha_rad, beta_rad, theta_rad)  # rebuild full state from the free vars
    u = np.array([de_rad, da_rad, dr_rad, np.clip(throttle_cmd, 0.0, 1.0)],   # control vector with throttle clip
                 dtype=np.float64)
    dx = b747_ode_6dof(x, u, 0.0, engine_out_params)                # evaluate 6-DoF ODE under engine-out damage
    return np.array([dx[0], dx[1], dx[2], dx[3], dx[4], dx[5], dx[11]], dtype=np.float64)  # need u̇,v̇,ẇ,ṗ,q̇,ṙ,ḣ = 0

z0 = np.array([trim_result.alpha_rad, math.radians(-0.44), trim_result.theta_rad,  # initial guess: from healthy trim
               delta_e_trim_rad, math.radians(-4.4), math.radians(-2.64),          # plus typical da/dr offsets
               throttle_engine_out_estimate], dtype=np.float64)                    # plus boosted throttle guess
engine_out_solution, info, ier, msg = fsolve(                                      # solve nonlinear residual = 0
    engine_out_trim_residual, z0, full_output=True, xtol=1e-10, maxfev=2_000,      # tight tolerance, generous budget
)
engine_out_residual_norm = float(np.linalg.norm(info["fvec"]))                     # final residual norm for diagnostics
assert ier == 1 and engine_out_residual_norm < 1e-6, msg                           # fail fast if not converged

(alpha_engine_out_rad, beta_engine_out_rad, theta_engine_out_rad,                  # unpack converged solution
 delta_e_engine_out_rad, delta_a_engine_out_rad, delta_r_engine_out_rad,
 throttle_engine_out) = engine_out_solution
throttle_engine_out = float(np.clip(throttle_engine_out, 0.0, 1.0))                # enforce [0,1] just in case
engine_out_trim_action = np.array([                                                # engine-out 4-channel trim action
    delta_e_engine_out_rad, delta_a_engine_out_rad,                                # de, da
    delta_r_engine_out_rad, throttle_engine_out,                                   # dr, throttle
], dtype=np.float64)

print(f"Healthy trim @ FL200, V={V_REF_FT_S:.0f} ft/s:")                           # report healthy trim
print(f"  alpha = theta = {theta_ref_deg:+.3f} deg")                               # AoA ≈ pitch in trim (gamma=0)
print(f"  delta_e_trim  = {math.degrees(delta_e_trim_rad):+.3f} deg, throttle_trim = {throttle_trim:.4f}")
print(f"Engine-out trim residual norm = {engine_out_residual_norm:.2e}")           # confirm small residual


## 3. Reference schedule — coordinated banked turn

Build a `reference_schedule(t)` returning a target dict `{V, h, theta, psi, phi, beta}`. The bank schedule is a trapezoidal ramp; the heading schedule is the analytic integral of the coordinated-turn rate ψ̇ ≈ g·tan(|φ|) / V over the same window.

Numbers (chosen for a clean-looking maneuver, not maximum aggression):
* Bank: peak +15° (right bank).
* Roll rate: ±1.5 °/s during ramps.
* Steady-turn ψ̇ ≈ g·tan(15°) / V ≈ 32.17·0.268 / 674 ≈ 0.0128 rad/s ≈ **0.732 °/s**.
* Total Δψ ≈ 3.62° (ramp-in) + 0.733·112.95° (steady) + 3.62° (roll-out) ≈ **90.0°**.

θ gets a small +0.3° offset during the turn to compensate for the lift-vector tilt at 15° bank (`L·cos(φ)`-style proxy). β reference stays at 0 in the un-failed phase and blends toward the engine-out β trim post-failure.


In [ ]:
G_FT_S2 = 32.17                                              # gravitational acceleration in ft/s^2

def _phi_ref_rad(t: float) -> float:                             # bank-angle reference at time t (radians)
    if t < ROLL_IN_S[0]:                                         # before roll-in: wings level
        return 0.0
    if t < ROLL_IN_S[1]:                                         # inside roll-in: linear ramp 0 -> peak
        frac = (t - ROLL_IN_S[0]) / (ROLL_IN_S[1] - ROLL_IN_S[0])
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) * frac
    if t < STEADY_S[1]:                                          # steady-turn window: hold peak bank
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG)
    if t < ROLL_OUT_S[1]:                                        # roll-out: linear ramp peak -> 0
        frac = 1.0 - (t - ROLL_OUT_S[0]) / (ROLL_OUT_S[1] - ROLL_OUT_S[0])
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) * frac
    return 0.0                                                    # after roll-out: wings level again

def _psi_dot_coordinated(phi_rad: float, V_ft_s: float) -> float:  # coordinated-turn heading rate
    # Coordinated turn: psi_dot = (g/V) * tan(phi).
    # Positive bank (right wing down) -> positive psi rate (right turn).
    return G_FT_S2 * math.tan(phi_rad) / max(V_ft_s, 1.0)         # max() guards against V near zero

def _phi_dot_ref_rad_s(t: float) -> float:                       # analytic d/dt of phi_ref (feed-forward to da)
    if ROLL_IN_S[0] <= t < ROLL_IN_S[1]:                         # inside roll-in: positive ramp slope
        return TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) / (ROLL_IN_S[1] - ROLL_IN_S[0])
    if ROLL_OUT_S[0] <= t < ROLL_OUT_S[1]:                       # inside roll-out: negative ramp slope
        return -TURN_BANK_SIGN * math.radians(TURN_BANK_DEG) / (ROLL_OUT_S[1] - ROLL_OUT_S[0])
    return 0.0                                                    # zero on level / steady-turn segments

# Pre-compute psi_ref(t) by trapezoidal integration of psi_dot(phi_ref(t), V_REF).
_t_grid = np.arange(0.0, TOTAL_TIME + DT, DT)                    # uniform time grid for integration
_phi_grid = np.array([_phi_ref_rad(float(t)) for t in _t_grid])  # phi_ref sampled on the grid
_psi_dot_grid = np.array([_psi_dot_coordinated(float(p), V_REF_FT_S) for p in _phi_grid])  # psi_dot on grid
_psi_grid = np.zeros_like(_t_grid)                               # accumulator for psi_ref(t)
for i in range(1, len(_t_grid)):                                 # trapezoidal rule integration
    _psi_grid[i] = _psi_grid[i-1] + 0.5 * DT * (_psi_dot_grid[i-1] + _psi_dot_grid[i])
PSI_FINAL_RAD = float(_psi_grid[int(ROLL_OUT_S[1] / DT)])        # final heading after roll-out (held thereafter)

def _psi_ref_rad(t: float) -> float:                             # piecewise heading reference
    if t <= 0.0:                                                 # before t=0: 0 heading
        return 0.0
    if t >= ROLL_OUT_S[1]:                                       # after roll-out: hold final heading
        return PSI_FINAL_RAD
    return float(np.interp(t, _t_grid, _psi_grid))               # linear interp on the precomputed grid

def _theta_offset_rad(t: float) -> float:                        # small pitch boost during banked turn
    # Small pitch/lift compensation that follows bank demand continuously.
    # This avoids a pitch-reference step during the healthy turn.
    peak_phi = max(math.radians(abs(TURN_BANK_DEG)), 1e-9)        # avoid division by zero in degenerate case
    bank_fraction = abs(_phi_ref_rad(t)) / peak_phi              # current |phi_ref| as fraction of peak
    return math.radians(0.3) * bank_fraction ** 2                # quadratic ramp -> +0.3 deg at full bank

def reference_schedule(t: float, beta_target_rad: float = 0.0) -> dict:  # public API used by the rollout
    phi_ref = _phi_ref_rad(t)                                    # cache phi_ref to reuse below
    return {                                                     # full target dict consumed by uftc_state_transform_dyn
        "V": V_REF_FT_S,                                         # speed setpoint (ft/s)
        "h": ALT_REF_FT,                                         # altitude setpoint (ft)
        "theta": float(trim_result.theta_rad) + _theta_offset_rad(t),  # pitch setpoint (rad), bank-compensated
        "psi": _psi_ref_rad(t),                                  # heading setpoint (rad)
        "phi": phi_ref,                                          # bank setpoint (rad)
        "beta": beta_target_rad,                                 # sideslip setpoint (rad), 0 in healthy phase
        "phi_dot": _phi_dot_ref_rad_s(t),                        # bank-rate feed-forward used by aileron law
        "psi_dot": _psi_dot_coordinated(phi_ref, V_REF_FT_S),    # heading-rate feed-forward used by rudder law
    }

# Visualise the reference
phi_ref_deg = np.degrees(_phi_grid)                              # bank-angle reference in degrees for plotting
psi_ref_deg = np.degrees(_psi_grid)                              # heading reference in degrees for plotting
psi_dot_deg_s = np.degrees(_psi_dot_grid)                        # heading rate in deg/s for plotting

fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)   # 3-row figure stacked on shared time axis
axes[0].plot(_t_grid, phi_ref_deg, color='tab:blue', lw=1.4)     # phi_ref(t)
axes[0].axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5, label='engine failure')  # damage marker
axes[0].set_ylabel('phi_ref, deg'); axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[1].plot(_t_grid, psi_ref_deg, color='tab:green', lw=1.4)    # psi_ref(t)
axes[1].axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5)
axes[1].set_ylabel('psi_ref, deg'); axes[1].grid(True, alpha=0.3)
axes[2].plot(_t_grid, psi_dot_deg_s, color='tab:purple', lw=1.4) # psi_dot_ref(t)
axes[2].axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5)
axes[2].set_xlabel('time, s'); axes[2].set_ylabel('psi_dot, deg/s'); axes[2].grid(True, alpha=0.3)
axes[0].set_title('Coordinated-turn reference schedule')
plt.tight_layout(); plt.show()                                   # render the reference figure
print(f'Total heading change Δψ over schedule = {math.degrees(PSI_FINAL_RAD):+.2f} deg')   # final Δψ
print(f'Steady-turn psi_dot at peak bank      = '                                          # peak rate sanity check
      f'{math.degrees(_psi_dot_coordinated(TURN_BANK_SIGN*math.radians(TURN_BANK_DEG), V_REF_FT_S)):+.3f} deg/s')


## 4. Damage profile — engine failure at t = 50 s

The stock `LEFT_OUTER_ENGINE_FAILURE` preset triggers the failure at t = 10 s. We construct a local `DamageProfile` with the same `EngineFailureEvent` parameters but `trigger_time=DAMAGE_TIME=50 s`.


In [ ]:
TURN_ENGINE_FAILURE = DamageProfile(                         # programmable damage timeline
    events=[                                                  # list of timed damage events (here just one)
        EngineFailureEvent(                                   # left-outer engine flame-out
            trigger_time=DAMAGE_TIME,                         # at exactly DAMAGE_TIME = 50 s
            engine_id=1,                                      # engine #1 = left outer (B-747 numbering)
            thrust_fraction=0.0,                              # remaining thrust fraction = 0 (full flame-out)
            label='left_outer_engine_flameout_mid_turn',      # human-readable label for logs
        ),
    ],
)
print(f'Damage profile: {TURN_ENGINE_FAILURE.events[0]}')      # echo the single event for sanity


## 5. UFTC controller and helpers

The UFTC state remains `[eV, eh, e_theta, e_psi, e_phi, e_beta, p, q, r]` — exactly as in the Phase 4 demo. The only change: `uftc_state_transform` now takes a **time-varying** setpoint `ref_t` instead of a frozen `ref0`. Everything downstream stays the same.


In [ ]:
UFTC_RESIDUAL_SCALE = np.array([                                  # residual-action scale: how much UFTC can nudge each channel
    0.0,                                                          # de: no residual (handled by state feedback)
    math.radians(0.05),                                           # da: ±0.05 deg residual aileron
    math.radians(0.05),                                           # dr: ±0.05 deg residual rudder
    0.0,                                                          # throttle: no residual
], dtype=np.float64)

UFTC_STATE_SCALE = np.array([                                     # normalisation scales for the UFTC state vector
    25.0, 250.0,                                                  # eV (ft/s), eh (ft)
    math.radians(3.0), math.radians(8.0),                         # e_theta (rad), e_psi (rad)
    math.radians(8.0), math.radians(2.0),                         # e_phi (rad), e_beta (rad)
    1.0, 1.0, 1.0,                                                # body rates p, q, r (rad/s)
], dtype=np.float64)
N_UFTC_STATE = int(UFTC_STATE_SCALE.size)                         # cache state dimension (= 9)
UFTC_OMEGA_INDICES = [6, 7, 8]                                    # indices of body rates (used by L2 inner loop)
UFTC_REFERENCE = np.zeros(N_UFTC_STATE, dtype=np.float64)         # zero-reference: maneuver enters via ref_t

UFTC_F_INIT = np.eye(N_UFTC_STATE, dtype=np.float64) * 0.99       # warm-start F: nearly identity (slight decay)
UFTC_F_INIT[2, 7] = DT / UFTC_STATE_SCALE[2]                      # theta integrates q with normalisation
UFTC_F_INIT[3, 8] = DT / UFTC_STATE_SCALE[3]                      # psi integrates r with normalisation
UFTC_F_INIT[4, 6] = DT / UFTC_STATE_SCALE[4]                      # phi integrates p with normalisation

UFTC_G_INIT = np.zeros((N_UFTC_STATE, 4), dtype=np.float64)       # warm-start B-matrix: zero baseline
UFTC_G_INIT[0] = [0.0, 0.0, 0.0, 0.08]                            # eV reacts mostly to throttle
UFTC_G_INIT[1] = [-0.02, 0.0, 0.0, 0.03]                          # eh reacts to elevator (-) and throttle (+)
UFTC_G_INIT[6] = [0.00, 0.30, 0.00, 0.00]                         # p reacts to aileron
UFTC_G_INIT[7] = [0.40, 0.00, 0.00, 0.10]                         # q reacts to elevator and (weakly) throttle
UFTC_G_INIT[8] = [0.00, 0.05, 0.30, 0.00]                         # r reacts mostly to rudder, a bit to aileron
UFTC_G_INNER = UFTC_G_INIT[UFTC_OMEGA_INDICES].copy()             # inner-loop slice (rates only) for AA-INDI

L4_ACTION_SCALE = 0.05                                            # L4 D-SAC outer action scale (small residual)
L4_TRIM_FREE_INDICES = {'V_idx': 0, 'gamma_idx': 1, 'alpha_idx': 2, 'q_idx': 7}  # trim-free indices fed to L4

# Phase 4 monitor calibration — relaxed d=(80,)*5 to keep the alarm meaningful
# during a long maneuver that already drives V_INDI/V_FDD higher.
MONITOR_CFG = dict(
    enable_monitor=True,                                          # turn the composite Lyapunov monitor on
    monitor_c_weights=(0.0, 0.4, 0.0, 0.3, 0.3),                  # weights of (V_hj, V_indi, V_iadp, V_dsac, V_fdd)
    monitor_d_disturbance=(80.0,) * 5,                            # disturbance budget per component
    monitor_alarm_warn_frac=0.4,                                  # WARN at 40% of mu_uub_pred
    monitor_alarm_critical_frac=0.7,                              # CRITICAL at 70% of mu_uub_pred
    monitor_cooldown_steps=20,                                    # cool-down between macro-action firings
)

def make_env(damage_profile=None, n_steps=N_EP + 5):              # construct a fresh nonlinear B-747 environment
    return NonlinearB747Env(
        trim_at=(ALT_REF_FT, V_REF_FT_S),                         # trim altitude and speed for env initialization
        number_time_steps=n_steps,                                # episode length
        dt=DT, integrator='rk4', action_space='virtual',          # 50 ms RK4 integrator, virtual control space
        config=B747Configuration.NOMINAL,                         # nominal aero/engine config
        damage_profile=damage_profile,                            # optional damage timeline
    )

def make_controller(*, enable_l4_outer=True, enable_trim_free=True,    # build a UFTCController with chosen flags
                    enable_monitor=True):
    cfg_kwargs = dict(
        dt=DT,                                                    # control step (matches env)
        fdd_warmup_steps=0,                                       # FDD active from t=0
        omega_indices=UFTC_OMEGA_INDICES,                         # which state indices are body rates
        middle_lookahead_dt=0.3,                                  # IADP look-ahead horizon (s)
        trust_radius_nominal=0.03,                                # trust radius in nominal (un-failed) regime
        trust_radius_fault=0.15,                                  # relaxed trust radius after fault declared
        fdd_cfg=FDDConfig(process_noise=1e-4, measurement_noise=1e-3,
                          adapt_Q=False, adapt_R=False, drift=6.0, h_alarm=15.0),  # FDD Kalman + alarm thresholds
        inner_cfg=AAINDIConfig(dt=DT, ref_wn=3.0, ref_zeta=0.9,    # L2 AA-INDI inner-loop config
                               u_magnitude_limit=1.0, u_rate_limit=5.0,
                               G_init=UFTC_G_INNER, ref_error_kp=2.0,
                               ref_error_ki=0.05, seed=0),
        middle_cfg=IADPConfig(dt=DT,                                # L3 IADP middle-loop config
                              Q=np.diag([20., 40., 20., 20., 20., 0.5, 2., 2., 2.]),  # state penalty (per channel)
                              R=np.diag([8., 8., 8., 20.]),                          # input penalty (per channel)
                              gamma=0.85, gamma_rls=0.995,                            # ADP discount + RLS forgetting
                              u_magnitude_limit=1.0, u_rate_limit=4.0,                # action / rate limits
                              policy_eval_every=80, policy_eval_blend=0.15),          # policy-eval cadence + blend
        enable_l4_outer=bool(enable_l4_outer),                    # toggle L4 D-SAC outer
        l4_action_scale=L4_ACTION_SCALE,                          # L4 action scale
        l4_eval_mode=True,                                        # eval mode: deterministic actor, no exploration
        l4_seed=0,                                                # L4 RNG seed
        l4_trim_free=(L4_TRIM_FREE_INDICES if enable_trim_free else None),  # which channels L4 may trim-free
    )
    if enable_monitor:                                            # optionally enable composite Lyapunov monitor
        cfg_kwargs.update(MONITOR_CFG)
    return UFTCController(                                        # finally build the controller
        n_state=N_UFTC_STATE, n_control=4,                        # 9 states, 4 controls
        nominal_F=UFTC_F_INIT, nominal_G=UFTC_G_INIT,             # warm-start identification matrices
        config=UFTCConfig(**cfg_kwargs),                          # assembled config
    )


In [ ]:
STATE_FEEDBACK_GAINS = {                                          # hand-tuned PI/PD gains for the envelope-allocator law
    'de_h': 2.5e-3,                                               # elevator response to altitude error
    'de_theta': 1.8,                                              # elevator response to pitch error
    'de_q': 1.0,                                                  # elevator damping on q
    'throttle_v': -1.2e-1,                                        # throttle response to speed error
    'throttle_h': -2.0e-4,                                        # throttle response to altitude error (small bias)
    'da_phi': -2.5,                                               # aileron response to bank error
    'da_p': -1.5,                                                 # aileron damping on p
    'da_phi_dot': 2.25,                                           # aileron feed-forward proportional to phi_dot_ref
    'dr_r': 3.0,                                                  # rudder damping on r
    'dr_psi': 1.5,                                                # rudder response to heading error
    'dr_beta': -0.5,                                              # rudder response to sideslip error
    'dr_psi_dot': -3.0,                                           # rudder feed-forward proportional to psi_dot_ref
}

def wrap_deg(angle_deg):                                          # wrap angle to (-180, 180] in degrees
    return (float(angle_deg) + 180.0) % 360.0 - 180.0

def wrap_rad(angle_rad):                                          # wrap angle to (-pi, pi] in radians
    return (float(angle_rad) + math.pi) % (2.0 * math.pi) - math.pi

def true_airspeed_ft_s(obs):                                      # ||(u, v, w)||_2 in ft/s
    obs = np.asarray(obs, dtype=np.float64).reshape(-1)           # ensure flat array
    return float(np.linalg.norm(obs[:3]))                         # body-axis velocity magnitude

def altitude_ft(obs):                                             # altitude in feet from NED z_e
    return float(-np.asarray(obs, dtype=np.float64).reshape(-1)[11])

def body_sideslip_rad(obs):                                       # sideslip beta from body velocity components
    obs = np.asarray(obs, dtype=np.float64).reshape(-1)
    v = max(true_airspeed_ft_s(obs), 1.0)                         # avoid div-by-zero on V close to 0
    return float(math.asin(np.clip(obs[1] / v, -1.0, 1.0)))       # beta = asin(v/V), clipped for safety

def engine_mu_from_info(info):                                    # extract per-engine mu (thrust multiplier) from env info
    damage_state = info.get('damage_state', {}) if isinstance(info, dict) else {}
    em = damage_state.get('engines_mu', {}) if isinstance(damage_state, dict) else {}
    return {eid: float(em.get(eid, em.get(str(eid), 1.0))) for eid in (1,2,3,4)}  # default 1.0 (healthy)

def engine_loss_from_info(info, fallback=0.0):                    # convert per-engine mu to scalar loss in [0,1]
    em = engine_mu_from_info(info)
    if not em:
        return float(fallback)                                    # info without engines_mu: keep previous estimate
    return float(np.clip(1.0 - em.get(1, 1.0), 0.0, 1.0))         # loss = 1 - mu of engine #1 (the failing one)

def per_engine_thrust_lb(obs, throttle, em):                      # back out per-engine thrust given throttle and mu
    h = altitude_ft(obs)                                          # altitude needed for ISA Mach computation
    mach = true_airspeed_ft_s(obs) / isa_speed_of_sound_ft_s(h)   # Mach number based on TAS / a(h)
    cluster = engine_model.installed_thrust(mach=mach, altitude_ft=h, throttle=throttle)  # 4-engine cluster thrust
    per = cluster / 4.0                                           # split evenly across 4 engines
    return np.array([per * float(em.get(eid, 1.0)) for eid in (1,2,3,4)], dtype=np.float64)  # scale by per-engine mu

def trim_action_for_loss(loss):                                   # blend healthy and engine-out trim by loss factor
    f = float(np.clip(loss, 0.0, 1.0))                            # ensure f in [0,1]
    return (1.0 - f) * healthy_trim_action + f * engine_out_trim_action  # convex combination

def clip_physical_action(action):                                 # enforce physical actuator limits
    a = np.asarray(action, dtype=np.float64).reshape(4)           # 4-vector view
    return np.array([
        np.clip(a[0], -math.radians(25.0), math.radians(25.0)),   # elevator: ±25 deg
        np.clip(a[1], -math.radians(20.0), math.radians(20.0)),   # aileron: ±20 deg
        np.clip(a[2], -math.radians(25.0), math.radians(25.0)),   # rudder:  ±25 deg
        np.clip(a[3], 0.0, 1.0),                                  # throttle: [0, 1]
    ], dtype=np.float64)

def uftc_state_transform_dyn(obs, ref_t, engine_loss_estimate):   # build UFTC state from raw obs against time-varying ref
    obs = np.asarray(obs, dtype=np.float64).reshape(-1)
    loss = float(np.clip(engine_loss_estimate, 0.0, 1.0))         # blend factor in [0,1]
    beta_target = (1.0 - loss) * ref_t['beta'] + loss * beta_engine_out_rad  # blend nominal beta with engine-out beta
    raw = np.array([                                              # build error-form state in physical units
        true_airspeed_ft_s(obs) - ref_t['V'],                     # eV = V_actual - V_ref
        altitude_ft(obs) - ref_t['h'],                            # eh = h_actual - h_ref
        wrap_rad(obs[7] - ref_t['theta']),                        # e_theta wrapped to (-pi,pi]
        wrap_rad(obs[8] - ref_t['psi']),                          # e_psi wrapped
        wrap_rad(obs[6] - ref_t['phi']),                          # e_phi wrapped
        body_sideslip_rad(obs) - beta_target,                     # e_beta = beta_actual - beta_target
        obs[3], obs[4], obs[5],                                   # body rates p, q, r
    ], dtype=np.float64)
    return raw / UFTC_STATE_SCALE                                 # normalise per-channel by the scale vector

def state_feedback_action_dyn(x_uftc, ref_t, engine_loss_estimate):  # envelope-allocator law (the "L0" baseline)
    x = np.asarray(x_uftc, dtype=np.float64).reshape(N_UFTC_STATE)   # x is the normalised UFTC state
    v_err, h_err = x[0]*UFTC_STATE_SCALE[0], x[1]*UFTC_STATE_SCALE[1]  # de-normalise back to ft/s, ft
    theta_err = x[2]*UFTC_STATE_SCALE[2]                              # de-normalise theta error
    psi_err = x[3]*UFTC_STATE_SCALE[3]                                # de-normalise psi error
    phi_err = x[4]*UFTC_STATE_SCALE[4]                                # de-normalise phi error
    beta_err = x[5]*UFTC_STATE_SCALE[5]                               # de-normalise beta error
    p, q, r = x[6], x[7], x[8]                                        # body rates already in rad/s
    phi_dot_ref = float(ref_t.get('phi_dot', 0.0))                    # bank-rate FF
    psi_dot_ref = float(ref_t.get('psi_dot', 0.0))                    # heading-rate FF
    action = trim_action_for_loss(engine_loss_estimate)               # start from blended trim (healthy or engine-out)
    action[0] += (STATE_FEEDBACK_GAINS['de_h']*h_err                  # elevator increment from h, theta, q
                  + STATE_FEEDBACK_GAINS['de_theta']*theta_err
                  + STATE_FEEDBACK_GAINS['de_q']*q)
    action[1] += (STATE_FEEDBACK_GAINS['da_phi']*phi_err              # aileron increment from phi, p, phi_dot_ref
                  + STATE_FEEDBACK_GAINS['da_p']*p
                  + STATE_FEEDBACK_GAINS['da_phi_dot']*phi_dot_ref)
    action[2] += (STATE_FEEDBACK_GAINS['dr_r']*r                      # rudder increment from r, psi, beta, psi_dot_ref
                  + STATE_FEEDBACK_GAINS['dr_psi']*psi_err
                  + STATE_FEEDBACK_GAINS['dr_beta']*beta_err
                  + STATE_FEEDBACK_GAINS['dr_psi_dot']*psi_dot_ref)
    action[3] += STATE_FEEDBACK_GAINS['throttle_v']*v_err + STATE_FEEDBACK_GAINS['throttle_h']*h_err  # throttle increment
    return clip_physical_action(action)                              # final physical clip on actuator limits

def compose_action(state_feedback, uftc_action_norm):                # combine state-feedback baseline with UFTC residual
    residual = UFTC_RESIDUAL_SCALE * np.clip(                        # rescale and clip the normalised UFTC output
        np.asarray(uftc_action_norm, dtype=np.float64).reshape(4), -1.0, 1.0)
    return clip_physical_action(np.asarray(state_feedback, dtype=np.float64).reshape(4) + residual)


## 5b. Classical PID baseline for comparison

The PID baseline tracks exactly the same coordinated-turn reference and uses the same physical actuator limits as the UFTC rollout. It is a fixed-gain, fixed-structure controller: the gains, trim point, and targets are set once before the rollout and are **not** changed after the engine failure. PID has no UFTC residual control, no adaptive allocation, no L4 D-SAC, no UUB monitor, and no engine-loss-aware trim scheduling.


In [ ]:
from tensoraerospace.agent.pid import PID                       # in-tree PID building block

PID_GAINS = {                                                     # gain set used for the PID baseline
    # PID.select_action uses error = setpoint - measurement.
    # Signs below are chosen for the B-747 virtual action convention:
    # negative elevator is nose-up, positive aileron/right bank follows phi sign.
    'de_h':        dict(kp=-2.5e-3, ki=-1.0e-6, kd=0.0),         # elevator vs altitude error
    'de_theta':    dict(kp=-1.8,    ki=-2.0e-2, kd=-1.0),         # elevator vs pitch error
    'throttle_v':  dict(kp=+1.2e-1, ki=+2.0e-4, kd=0.0),         # throttle vs speed error
    'throttle_h':  dict(kp=+2.0e-4, ki=+2.0e-7, kd=0.0),         # throttle vs altitude error
    'da_phi':      dict(kp=+2.5,    ki=+1.0e-2, kd=+1.5),         # aileron vs bank error
    'dr_psi_err':  dict(kp=-1.5,    ki=-4.0e-3, kd=-3.0),         # rudder vs heading error
    'dr_beta':     dict(kp=+0.5,    ki=+2.0e-3, kd=0.0),         # rudder vs sideslip error
}
PID_FEEDFORWARD_GAINS = {                                         # PID-side feed-forward gains
    'da_phi_dot': 2.25,                                           # aileron FF on phi_dot_ref (matches UFTC FF)
}
PID_INCREMENT_LIMITS = {                                          # per-channel increment caps to keep PID safe
    'de': math.radians(18.0),
    'da': math.radians(16.0),
    'dr': math.radians(20.0),
    'throttle': 0.35,
}

class B747TurnPIDController:
    '''Classical PID turn controller used only as a comparison baseline.'''

    def __init__(self, gains=None, feedforward_gains=None):       # allow overriding gains for tuning
        self.gains = dict(PID_GAINS if gains is None else gains)
        self.feedforward_gains = dict(PID_FEEDFORWARD_GAINS if feedforward_gains is None else feedforward_gains)
        self._make_axes()                                         # instantiate one PID per scalar control axis

    def _make_axes(self):                                         # build all underlying PID instances
        self.de_h = PID(env=None, dt=DT, **self.gains['de_h'])
        self.de_theta = PID(env=None, dt=DT, **self.gains['de_theta'])
        self.throttle_v = PID(env=None, dt=DT, **self.gains['throttle_v'])
        self.throttle_h = PID(env=None, dt=DT, **self.gains['throttle_h'])
        self.da_phi = PID(env=None, dt=DT, **self.gains['da_phi'])
        self.dr_psi_err = PID(env=None, dt=DT, **self.gains['dr_psi_err'])
        self.dr_beta = PID(env=None, dt=DT, **self.gains['dr_beta'])
        self.axes = [
            self.de_h, self.de_theta, self.throttle_v, self.throttle_h,
            self.da_phi, self.dr_psi_err, self.dr_beta,
        ]

    def reset(self):                                              # zero integrator/derivative state on every axis
        for axis in self.axes:
            axis.reset()

    def predict(self, obs, ref_t):                                # one-step control given current obs and reference
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        beta_target = float(ref_t['beta'])                        # PID always uses nominal beta=0 (no engine-out blend)

        v_actual = true_airspeed_ft_s(obs)                        # current TAS
        h_actual = altitude_ft(obs)                               # current altitude
        theta_actual = float(obs[7])                              # current pitch
        phi_actual = float(obs[6])                                # current bank
        beta_actual = body_sideslip_rad(obs)                      # current sideslip
        psi_err = wrap_rad(float(obs[8]) - float(ref_t['psi']))   # wrapped heading error

        # Fixed PID baseline: no engine-loss estimate and no post-failure trim scheduling.
        base = healthy_trim_action.copy()                         # always trim at healthy cruise
        de_increment = (
            self.de_h.select_action(ref_t['h'], h_actual)         # PID on altitude
            + self.de_theta.select_action(ref_t['theta'], theta_actual)  # PID on pitch
        )
        da_increment = (
            self.da_phi.select_action(ref_t['phi'], phi_actual)   # PID on bank
            + self.feedforward_gains['da_phi_dot'] * float(ref_t.get('phi_dot', 0.0))  # FF on phi_dot
        )
        dr_increment = (
            self.dr_psi_err.select_action(0.0, psi_err)           # PID on heading error
            + self.dr_beta.select_action(beta_target, beta_actual)  # PID on sideslip
        )
        throttle_increment = (
            self.throttle_v.select_action(ref_t['V'], v_actual)   # PID on speed
            + self.throttle_h.select_action(ref_t['h'], h_actual) # PID on altitude (small bias)
        )

        increments = np.array([                                   # collect and clip per-channel increments
            np.clip(de_increment, -PID_INCREMENT_LIMITS['de'], PID_INCREMENT_LIMITS['de']),
            np.clip(da_increment, -PID_INCREMENT_LIMITS['da'], PID_INCREMENT_LIMITS['da']),
            np.clip(dr_increment, -PID_INCREMENT_LIMITS['dr'], PID_INCREMENT_LIMITS['dr']),
            np.clip(throttle_increment, -PID_INCREMENT_LIMITS['throttle'], PID_INCREMENT_LIMITS['throttle']),
        ], dtype=np.float64)
        return clip_physical_action(base + increments)            # final physical-limit clip


def make_pid_controller():                                        # convenience factory mirroring make_controller()
    return B747TurnPIDController()


## 6. Pre-trained L4 D-SAC weights (optional)

Load `actor.pt` / `critic*.pt` / `target*.pt` from `artifacts/dsac/b747_engine_out_v1/` if available, otherwise fall back to a random-init actor with a printed note.

**Caveat**: those weights were trained on the stationary-trim engine-out task, *not* on a banked turn. The actor will give degraded performance during the maneuver — that's an honest result and a curriculum-extension argument, not a stack failure.


In [ ]:
from tensoraerospace.agent.uftc.l4.dsac import DSACOuter        # L4 D-SAC outer-loop class

_REL = Path('artifacts/dsac/b747_engine_out_v1')                  # canonical relative path
_candidates = [_REL]                                              # build candidate list to handle any cwd
_cwd = Path.cwd().resolve()                                       # absolute current working directory
for parent in [_cwd, *_cwd.parents]:                              # walk up the tree looking for the artifacts dir
    cand = parent / _REL
    if cand not in _candidates:
        _candidates.append(cand)
weights_path = None                                               # final resolved path (None if not found)
for cand in _candidates:                                          # pick the first existing candidate that has actor.pt
    if cand.exists() and (cand / 'actor.pt').exists():
        weights_path = cand
        break
PRETRAINED_AVAILABLE = weights_path is not None                   # boolean flag used downstream

def _load_pretrained_into(ctl_target):                            # copy weights from a fresh DSACOuter into ctl.l4
    pretrained = DSACOuter.from_pretrained(weights_path)
    ctl_target.l4.actor.load_state_dict(pretrained.actor.state_dict())     # actor weights
    ctl_target.l4.critic1.load_state_dict(pretrained.critic1.state_dict()) # critic 1
    ctl_target.l4.critic2.load_state_dict(pretrained.critic2.state_dict()) # critic 2
    ctl_target.l4.target1.load_state_dict(pretrained.target1.state_dict()) # target 1 (Polyak)
    ctl_target.l4.target2.load_state_dict(pretrained.target2.state_dict()) # target 2 (Polyak)

if PRETRAINED_AVAILABLE:                                          # report status
    print(f'Pre-trained L4 weights found at {weights_path}')
else:
    print('No pre-trained L4 weights found - using random-init actor (eval_mode demonstration)')


## 7. Closed-loop rollout helper

`run_uftc_episode_turn(damage_profile, ctl)` follows the Phase 4 rollout pattern but feeds the time-varying reference into `uftc_state_transform_dyn` and `state_feedback_action_dyn`. UFTC's `reference` argument stays at zero; the time-varying reference enters through the envelope-allocator setpoint.


In [ ]:
def run_uftc_episode_turn(damage_profile, *, ctl):
    env = make_env(damage_profile=damage_profile)                 # fresh env per run
    obs, _ = env.reset()                                          # initial observation at trim
    ctl.reset()                                                   # zero out controller internal state
    engine_loss_estimate = 0.0                                    # FDD-derived scalar engine-loss estimate

    log_keys = [                                                  # exhaustive list of keys we will log per step
        'V', 'h', 'theta', 'psi', 'phi', 'beta',                  # tracking errors (actual - reference)
        'V_actual', 'h_actual', 'theta_actual', 'psi_actual', 'phi_actual', 'beta_actual',  # raw measurements
        'x_e', 'y_e',                                             # ground-track in NED ft
        'V_ref', 'h_ref', 'theta_ref', 'psi_ref', 'phi_ref', 'beta_ref',  # references at log time
        'p', 'q', 'r',                                            # body rates (deg/s for plotting)
        'de', 'da', 'dr', 'throttle',                             # commanded actuator outputs
        'engine_loss_estimate',                                   # FDD severity in [0,1]
        'T1','T2','T3','T4','T_total',                            # per-engine and total thrust
        'severity', 'fault_present', 'rls_gamma', 'innovation_norm',  # FDD diagnostics
        'l4_beta', 'l4_replay_size', 'l4_r_eff_norm',             # L4 D-SAC diagnostics
        'V_total', 'V_hj', 'V_indi', 'V_iadp', 'V_dsac', 'V_fdd', # composite Lyapunov components
        'alarm_level', 'mu_uub_pred',                             # monitor alarm and mu prediction
    ]
    logs = {k: np.zeros(N_EP, dtype=np.float64) for k in log_keys}  # preallocate buffers
    t_axis = np.arange(N_EP, dtype=np.float64) * DT               # global time axis
    t_log_axis = np.zeros(N_EP, dtype=np.float64)                 # actual log time per row (post-step)
    macro_events = []                                             # record of monitor-fired macro-actions
    alarm_trans = []                                              # record of alarm-state transitions
    prev_alarm = 'OK'                                             # initial alarm state

    for k in range(N_EP - 2):                                     # main step loop, leave a 2-step margin
        t_now = t_axis[k]                                         # current sim time
        ref_t = reference_schedule(float(t_now))                  # references at t_now
        x_uftc = uftc_state_transform_dyn(obs, ref_t, engine_loss_estimate)  # UFTC normalised state
        feedback_action = state_feedback_action_dyn(x_uftc, ref_t, engine_loss_estimate)  # baseline action
        u_norm = ctl.predict(x_uftc, UFTC_REFERENCE, time_step=k) # UFTC residual prediction (normalised)
        action = compose_action(feedback_action, u_norm)          # compose final physical action

        obs, _, _, trunc, info = env.step(action)                 # step environment
        engine_loss_estimate = engine_loss_from_info(info, fallback=engine_loss_estimate)  # update loss estimate
        t_log = float(t_axis[k+1] if k+1 < N_EP else t_now + DT)  # post-step time for logging
        ref_log = reference_schedule(t_log)                       # reference at post-step time
        next_x_uftc = uftc_state_transform_dyn(obs, ref_log, engine_loss_estimate)  # post-step UFTC state
        ctl.learn(next_x_uftc, UFTC_REFERENCE, time_step=k)       # one online learning step (RLS / replay / etc.)

        diag = ctl.diagnostics()                                  # pull controller diagnostics
        l4_diag = diag.get('l4', {})                              # L4 sub-diagnostics
        mon_out = getattr(ctl, '_monitor_out', None)              # monitor output, if monitor enabled
        if mon_out is not None:                                   # extract Lyapunov components
            v_hj = float(mon_out.components.V_hj)
            v_indi = float(mon_out.components.V_indi)
            v_iadp = float(mon_out.components.V_iadp)
            v_dsac = float(mon_out.components.V_dsac)
            v_fdd = float(mon_out.components.V_fdd)
            v_total = float(mon_out.V_total)                      # weighted sum of components
            mu_pred = float(mon_out.mu_uub_pred)                  # predicted UUB radius
            cur_alarm = str(mon_out.alarm)                        # 'OK' / 'WARN' / 'CRITICAL'
            for ma in mon_out.interventions:                      # record any macro-actions that fired this step
                macro_events.append((k, t_axis[k], str(ma.kind), dict(ma.payload)))
            if cur_alarm != prev_alarm:                           # record alarm-state transitions
                alarm_trans.append((k, t_axis[k], prev_alarm, cur_alarm))
                prev_alarm = cur_alarm
        else:                                                     # monitor disabled (Phase 1 / PID-equiv): zero everything
            v_hj = v_indi = v_iadp = v_dsac = v_fdd = 0.0
            v_total = 0.0; mu_pred = 0.0; cur_alarm = 'OK'
        alarm_int = {'OK': 0, 'WARN': 1, 'CRITICAL': 2}.get(cur_alarm, 0)  # numeric alarm level

        v_actual = true_airspeed_ft_s(obs)                        # log-time TAS
        h_actual = altitude_ft(obs)                               # log-time altitude
        theta_actual = math.degrees(obs[7])                       # pitch in degrees
        psi_actual = wrap_deg(math.degrees(obs[8]))               # heading in degrees, wrapped
        phi_actual = math.degrees(obs[6])                         # bank in degrees
        beta_actual = math.degrees(body_sideslip_rad(obs))        # sideslip in degrees
        x_e_actual = float(obs[9])                                # NED north position (ft)
        y_e_actual = float(obs[10])                               # NED east position (ft)
        beta_ref_effective_rad = ((1.0 - engine_loss_estimate) * ref_log['beta']     # blend nominal and engine-out
                                  + engine_loss_estimate * beta_engine_out_rad)
        beta_ref_effective_deg = math.degrees(beta_ref_effective_rad)

        # Log the post-step state at t + dt and compare it with the reference at the same time.
        t_log_axis[k] = t_log                                     # store the actual log timestamp
        logs['V'][k] = v_actual - ref_log['V']                    # tracking error: V
        logs['h'][k] = h_actual - ref_log['h']                    # tracking error: h
        logs['theta'][k] = math.degrees(wrap_rad(obs[7] - ref_log['theta']))  # wrapped theta error in deg
        logs['psi'][k] = math.degrees(wrap_rad(obs[8] - ref_log['psi']))      # wrapped psi error in deg
        logs['phi'][k] = math.degrees(wrap_rad(obs[6] - ref_log['phi']))      # wrapped phi error in deg
        logs['beta'][k] = beta_actual - beta_ref_effective_deg                 # beta error against blended ref

        logs['V_actual'][k] = v_actual                            # raw measurements
        logs['h_actual'][k] = h_actual
        logs['theta_actual'][k] = theta_actual
        logs['psi_actual'][k] = psi_actual
        logs['phi_actual'][k] = phi_actual
        logs['beta_actual'][k] = beta_actual
        logs['x_e'][k] = x_e_actual
        logs['y_e'][k] = y_e_actual
        logs['V_ref'][k] = ref_log['V']                           # references at log time
        logs['h_ref'][k] = ref_log['h']
        logs['theta_ref'][k] = math.degrees(ref_log['theta'])
        logs['psi_ref'][k] = math.degrees(ref_log['psi'])
        logs['phi_ref'][k] = math.degrees(ref_log['phi'])
        logs['beta_ref'][k] = beta_ref_effective_deg
        logs['p'][k] = math.degrees(obs[3])                       # body rates in deg/s
        logs['q'][k] = math.degrees(obs[4])
        logs['r'][k] = math.degrees(obs[5])
        logs['de'][k] = math.degrees(action[0])                   # actuator commands in deg
        logs['da'][k] = math.degrees(action[1])
        logs['dr'][k] = math.degrees(action[2])
        logs['throttle'][k] = float(action[3])                    # throttle in [0,1]
        logs['engine_loss_estimate'][k] = engine_loss_estimate
        thr = per_engine_thrust_lb(obs, float(action[3]), engine_mu_from_info(info))  # per-engine thrust
        logs['T1'][k], logs['T2'][k], logs['T3'][k], logs['T4'][k] = thr
        logs['T_total'][k] = float(np.sum(thr))                   # total cluster thrust
        logs['severity'][k] = float(diag['severity'])             # FDD severity output
        logs['fault_present'][k] = float(diag['fault_present'])   # FDD binary fault flag
        logs['rls_gamma'][k] = float(diag['rls_gamma'])           # FDD adaptive forgetting gamma
        logs['innovation_norm'][k] = float(diag['innovation_norm'])  # FDD residual innovation magnitude
        logs['l4_beta'][k] = float(l4_diag.get('beta_t', 0.0))    # L4 entropy weight
        logs['l4_replay_size'][k] = float(l4_diag.get('replay_size', 0))  # replay buffer size
        last_r_eff = getattr(ctl, '_last_r_eff', None)            # L4 effective reward (if exposed)
        if last_r_eff is not None:
            logs['l4_r_eff_norm'][k] = float(np.linalg.norm(last_r_eff))
        logs['V_total'][k] = v_total                              # composite Lyapunov total
        logs['V_hj'][k] = v_hj                                    # Hamilton-Jacobi component
        logs['V_indi'][k] = v_indi                                # AA-INDI component
        logs['V_iadp'][k] = v_iadp                                # IADP component
        logs['V_dsac'][k] = v_dsac                                # D-SAC component
        logs['V_fdd'][k] = v_fdd                                  # FDD component
        logs['alarm_level'][k] = float(alarm_int)
        logs['mu_uub_pred'][k] = mu_pred
        if trunc:                                                 # honor early termination if env returns trunc=True
            break

    n = k + 1                                                     # number of valid samples
    out = {key: vals[:n] for key, vals in logs.items()}           # truncate logs to n
    out['t'] = t_log_axis[:n]                                     # attach time vector
    out['macro_events'] = macro_events                            # macro-action firing log
    out['alarm_transitions'] = alarm_trans                        # alarm-state transition log
    return out


In [ ]:
def run_pid_episode_turn(damage_profile, *, pid_ctl):
    env = make_env(damage_profile=damage_profile)                 # fresh env, may include damage timeline
    obs, _ = env.reset()                                          # start at trim
    pid_ctl.reset()                                               # zero PID integrators / derivative state
    engine_loss_estimate = 0.0                                    # tracked here only for log compatibility

    log_keys = [                                                  # match the UFTC log schema for plot reuse
        'V', 'h', 'theta', 'psi', 'phi', 'beta',
        'V_actual', 'h_actual', 'theta_actual', 'psi_actual', 'phi_actual', 'beta_actual',
        'x_e', 'y_e',
        'V_ref', 'h_ref', 'theta_ref', 'psi_ref', 'phi_ref', 'beta_ref',
        'p', 'q', 'r',
        'de', 'da', 'dr', 'throttle',
        'engine_loss_estimate',
        'T1','T2','T3','T4','T_total',
        'severity', 'fault_present', 'rls_gamma', 'innovation_norm',
        'l4_beta', 'l4_replay_size', 'l4_r_eff_norm',
        'V_total', 'V_hj', 'V_indi', 'V_iadp', 'V_dsac', 'V_fdd',
        'alarm_level', 'mu_uub_pred',
    ]
    logs = {k: np.zeros(N_EP, dtype=np.float64) for k in log_keys}  # preallocate
    t_axis = np.arange(N_EP, dtype=np.float64) * DT
    t_log_axis = np.zeros(N_EP, dtype=np.float64)

    for k in range(N_EP - 2):                                     # main step loop
        t_now = t_axis[k]
        ref_t = reference_schedule(float(t_now))                  # PID consumes the same time-varying reference
        action = pid_ctl.predict(obs, ref_t)                      # PID one-shot action

        obs, _, _, trunc, info = env.step(action)
        engine_loss_estimate = engine_loss_from_info(info, fallback=engine_loss_estimate)  # for plotting only
        t_log = float(t_axis[k+1] if k+1 < N_EP else t_now + DT)
        ref_log = reference_schedule(t_log)

        v_actual = true_airspeed_ft_s(obs)
        h_actual = altitude_ft(obs)
        theta_actual = math.degrees(obs[7])
        psi_actual = wrap_deg(math.degrees(obs[8]))
        phi_actual = math.degrees(obs[6])
        beta_actual = math.degrees(body_sideslip_rad(obs))
        beta_ref_effective_deg = math.degrees(ref_log['beta'])    # PID never blends, so effective ref = nominal

        t_log_axis[k] = t_log
        logs['V'][k] = v_actual - ref_log['V']
        logs['h'][k] = h_actual - ref_log['h']
        logs['theta'][k] = math.degrees(wrap_rad(obs[7] - ref_log['theta']))
        logs['psi'][k] = math.degrees(wrap_rad(obs[8] - ref_log['psi']))
        logs['phi'][k] = math.degrees(wrap_rad(obs[6] - ref_log['phi']))
        logs['beta'][k] = beta_actual - beta_ref_effective_deg

        logs['V_actual'][k] = v_actual                            # raw measurements (same fields as UFTC log)
        logs['h_actual'][k] = h_actual
        logs['theta_actual'][k] = theta_actual
        logs['psi_actual'][k] = psi_actual
        logs['phi_actual'][k] = phi_actual
        logs['beta_actual'][k] = beta_actual
        logs['x_e'][k] = float(obs[9])
        logs['y_e'][k] = float(obs[10])
        logs['V_ref'][k] = ref_log['V']
        logs['h_ref'][k] = ref_log['h']
        logs['theta_ref'][k] = math.degrees(ref_log['theta'])
        logs['psi_ref'][k] = math.degrees(ref_log['psi'])
        logs['phi_ref'][k] = math.degrees(ref_log['phi'])
        logs['beta_ref'][k] = beta_ref_effective_deg
        logs['p'][k] = math.degrees(obs[3])
        logs['q'][k] = math.degrees(obs[4])
        logs['r'][k] = math.degrees(obs[5])
        logs['de'][k] = math.degrees(action[0])
        logs['da'][k] = math.degrees(action[1])
        logs['dr'][k] = math.degrees(action[2])
        logs['throttle'][k] = float(action[3])
        logs['engine_loss_estimate'][k] = engine_loss_estimate
        thr = per_engine_thrust_lb(obs, float(action[3]), engine_mu_from_info(info))
        logs['T1'][k], logs['T2'][k], logs['T3'][k], logs['T4'][k] = thr
        logs['T_total'][k] = float(np.sum(thr))
        logs['fault_present'][k] = float(engine_loss_estimate > 0.0)  # synthetic flag from loss estimate
        logs['severity'][k] = engine_loss_estimate
        if trunc:
            break

    n = k + 1
    out = {key: vals[:n] for key, vals in logs.items()}
    out['t'] = t_log_axis[:n]
    out['macro_events'] = []                                      # PID has no macro-actions
    out['alarm_transitions'] = []                                 # PID has no monitor
    return out


In [ ]:
def run_open_loop_episode_turn(damage_profile):
    # TRUE open-loop rollout: aircraft holds the healthy cruise trim throughout.
    # NO state feedback, NO turn-schedule pilot input, NO UFTC adaptation, NO
    # engine-loss compensation. When the engine fails mid-flight, the aircraft
    # simply diverges — yaw/roll moments from the asymmetric thrust propagate
    # freely with no controller reaction. The reference schedule is still
    # recorded in the log so the 3D viewer chart overlay shows what the aircraft
    # *should* be doing vs. what it actually does (i.e., nothing).
    env = make_env(damage_profile=damage_profile)                 # fresh env (with the same damage timeline)
    obs, _ = env.reset()                                          # start at trim

    # Static healthy-trim action — no maneuver, no feedback.
    nominal_action = np.array([                                   # constant 4-vector applied for the whole episode
        float(trim_result.elevator_rad),                          # elevator at healthy trim
        0.0,                                                      # aileron = 0
        0.0,                                                      # rudder = 0
        float(trim_result.throttle),                              # throttle at healthy trim
    ], dtype=np.float64)

    log_keys = [                                                  # mirror the UFTC schema (most fields stay zero)
        'V', 'h', 'theta', 'psi', 'phi', 'beta',
        'V_actual', 'h_actual', 'theta_actual', 'psi_actual', 'phi_actual', 'beta_actual',
        'x_e', 'y_e',
        'V_ref', 'h_ref', 'theta_ref', 'psi_ref', 'phi_ref', 'beta_ref',
        'p', 'q', 'r',
        'de', 'da', 'dr', 'throttle',
        'engine_loss_estimate',
        'T1', 'T2', 'T3', 'T4', 'T_total',
        'severity', 'fault_present', 'rls_gamma', 'innovation_norm',
        'l4_beta', 'l4_replay_size', 'l4_r_eff_norm',
        'V_total', 'V_hj', 'V_indi', 'V_iadp', 'V_dsac', 'V_fdd',
        'alarm_level', 'mu_uub_pred',
    ]
    logs = {k: np.zeros(N_EP, dtype=np.float64) for k in log_keys}
    t_axis = np.arange(N_EP, dtype=np.float64) * DT
    t_log_axis = np.zeros(N_EP, dtype=np.float64)

    for k in range(N_EP - 2):                                     # roll the aircraft forward without feedback
        t_now = t_axis[k]
        obs, _, _, trunc, info = env.step(nominal_action)         # apply the static healthy-trim action
        t_log = float(t_axis[k+1] if k+1 < N_EP else t_now + DT)

        v_actual = true_airspeed_ft_s(obs)                        # log-time TAS
        h_actual = altitude_ft(obs)                               # altitude
        theta_actual = math.degrees(obs[7])                       # pitch in deg
        psi_actual = wrap_deg(math.degrees(obs[8]))               # heading in deg
        phi_actual = math.degrees(obs[6])                         # bank in deg
        beta_actual = math.degrees(body_sideslip_rad(obs))        # sideslip in deg

        ref_log = reference_schedule(t_log)                       # reference is recorded only for the 3D overlay
        logs['V_actual'][k]     = v_actual
        logs['h_actual'][k]     = h_actual
        logs['theta_actual'][k] = theta_actual
        logs['psi_actual'][k]   = psi_actual
        logs['phi_actual'][k]   = phi_actual
        logs['beta_actual'][k]  = beta_actual
        logs['x_e'][k] = float(obs[9])                            # ground-track north
        logs['y_e'][k] = float(obs[10])                           # ground-track east

        logs['V_ref'][k]     = V_REF_FT_S                         # constant cruise reference for V
        logs['h_ref'][k]     = ALT_REF_FT                         # constant cruise reference for h
        logs['theta_ref'][k] = math.degrees(ref_log['theta'])     # actual theta_ref from schedule (for display)
        logs['psi_ref'][k]   = math.degrees(ref_log['psi'])       # actual psi_ref from schedule (for display)
        logs['phi_ref'][k]   = math.degrees(ref_log['phi'])       # actual phi_ref from schedule (for display)
        logs['beta_ref'][k]  = 0.0                                # beta_ref always 0 in the open-loop log

        logs['p'][k] = math.degrees(obs[3])                       # body rates in deg/s
        logs['q'][k] = math.degrees(obs[4])
        logs['r'][k] = math.degrees(obs[5])
        logs['de'][k] = math.degrees(nominal_action[0])           # echo the constant trim command
        logs['da'][k] = math.degrees(nominal_action[1])
        logs['dr'][k] = math.degrees(nominal_action[2])
        logs['throttle'][k] = float(nominal_action[3])

        t_log_axis[k] = t_log
        if trunc:
            break

    n_logged = int(np.argmax(t_log_axis == 0)) if t_log_axis[-1] == 0 else N_EP  # find last non-zero index
    n_logged = min(n_logged if n_logged > 0 else N_EP, N_EP)      # clamp to N_EP for safety
    logs['t'] = t_log_axis[:n_logged]
    for key in log_keys:
        logs[key] = logs[key][:n_logged]                          # truncate every series consistently
    return logs

log_open_loop = run_open_loop_episode_turn(TURN_ENGINE_FAILURE)   # actually run the open-loop rollout
print(f'Open-loop rollout: {len(log_open_loop["t"])} steps, '
      f'final t = {log_open_loop["t"][-1]:.2f} s')
print(f'  final psi error: {log_open_loop["psi_actual"][-1] - log_open_loop["psi_ref"][-1]:+.2f} deg')
print(f'  final phi error: {log_open_loop["phi_actual"][-1] - log_open_loop["phi_ref"][-1]:+.2f} deg')
print(f'  final V error:   {log_open_loop["V_actual"][-1] - log_open_loop["V_ref"][-1]:+.2f} ft/s')
print(f'  final h error:   {log_open_loop["h_actual"][-1] - log_open_loop["h_ref"][-1]:+.2f} ft')


## 8. Run PID and UFTC configurations

* **Phase 1**: `enable_l4_outer=False, enable_monitor=False` — L1 shield disabled (placeholder), L2 + L3 + envelope allocator only.
* **Phase 4**: full stack + pre-trained L4 + composite Lyapunov monitor with relaxed `d=(80,)*5` calibration.


In [ ]:
# Healthy baseline rollouts are used to separate nominal tracking lag from
# fault-induced deviation. With deterministic dynamics, damaged and healthy
# trajectories coincide until DAMAGE_TIME.
pid_nominal_ctl = make_pid_controller()                          # fresh PID for the nominal (no-damage) baseline
log_pid_nominal = run_pid_episode_turn(None, pid_ctl=pid_nominal_ctl)  # PID without damage = baseline
print(f'PID healthy baseline: {len(log_pid_nominal["t"])} steps, '
      f'final t = {log_pid_nominal["t"][-1]:.2f} s')

pid_ctl = make_pid_controller()                                  # fresh PID for the damaged rollout
log_pid = run_pid_episode_turn(TURN_ENGINE_FAILURE, pid_ctl=pid_ctl)  # PID with engine failure
print(f'PID damaged rollout: {len(log_pid["t"])} steps, final t = {log_pid["t"][-1]:.2f} s')

ctl_p1_nominal = make_controller(enable_l4_outer=False, enable_trim_free=False,    # Phase 1, no damage
                                 enable_monitor=False)
log_p1_nominal = run_uftc_episode_turn(None, ctl=ctl_p1_nominal)
print(f'Phase 1 healthy baseline: {len(log_p1_nominal["t"])} steps, '
      f'final t = {log_p1_nominal["t"][-1]:.2f} s')

ctl_p1 = make_controller(enable_l4_outer=False, enable_trim_free=False,            # Phase 1, with damage
                         enable_monitor=False)
log_p1 = run_uftc_episode_turn(TURN_ENGINE_FAILURE, ctl=ctl_p1)
print(f'Phase 1 damaged rollout: {len(log_p1["t"])} steps, final t = {log_p1["t"][-1]:.2f} s')

ctl_p4_nominal = make_controller(enable_l4_outer=True, enable_trim_free=True,      # Phase 4, no damage
                                  enable_monitor=True)
if PRETRAINED_AVAILABLE:                                                            # load pre-trained L4 if present
    _load_pretrained_into(ctl_p4_nominal)
log_p4_nominal = run_uftc_episode_turn(None, ctl=ctl_p4_nominal)
print(f'Phase 4 healthy baseline: {len(log_p4_nominal["t"])} steps, '
      f'final t = {log_p4_nominal["t"][-1]:.2f} s')

ctl_p4 = make_controller(enable_l4_outer=True, enable_trim_free=True,              # Phase 4, with damage
                         enable_monitor=True)
if PRETRAINED_AVAILABLE:
    _load_pretrained_into(ctl_p4)
    print(f'Phase 4: loaded pre-trained L4 weights from {weights_path}')
else:
    print('Phase 4: random-init L4 actor (no pre-trained weights)')
if ctl_p4.monitor is not None:
    print(f'Phase 4 monitor mu_uub_pred = {ctl_p4.monitor.mu_uub_pred:.4f}')        # report initial mu prediction
log_p4 = run_uftc_episode_turn(TURN_ENGINE_FAILURE, ctl=ctl_p4)
print(f'Phase 4 damaged rollout: {len(log_p4["t"])} steps, final t = {log_p4["t"][-1]:.2f} s')

def _aligned_len(log, baseline):                                  # smallest common log length for direct subtraction
    return min(len(log['t']), len(baseline['t']))

def common_t(log, baseline):                                      # aligned time vector for plotting
    n = _aligned_len(log, baseline)
    return np.asarray(log['t'][:n], dtype=np.float64)

def operational_target(log, baseline, key):                       # operational reference = healthy-baseline trajectory
    n = _aligned_len(log, baseline)
    t = np.asarray(log['t'][:n], dtype=np.float64)
    if key == 'V':
        return np.asarray(baseline['V_actual'][:n], dtype=np.float64)
    if key == 'h':
        return np.asarray(baseline['h_actual'][:n], dtype=np.float64)
    if key == 'theta':
        return np.asarray(baseline['theta_actual'][:n], dtype=np.float64)
    if key == 'psi':
        return np.asarray(baseline['psi_actual'][:n], dtype=np.float64)
    if key == 'phi':
        return np.asarray(baseline['phi_actual'][:n], dtype=np.float64)
    if key == 'beta':
        target = np.asarray(baseline['beta_actual'][:n], dtype=np.float64).copy()
        target[t >= DAMAGE_TIME] = np.asarray(log['beta_ref'][:n], dtype=np.float64)[t >= DAMAGE_TIME]
        return target                                             # post-damage: use the engine-out-blended reference
    raise KeyError(key)

def operational_deviation(log, baseline, key):                    # operational deviation = actual - operational target
    n = _aligned_len(log, baseline)
    target = operational_target(log, baseline, key)
    if key == 'V':
        return np.asarray(log['V_actual'][:n] - target, dtype=np.float64)
    if key == 'h':
        return np.asarray(log['h_actual'][:n] - target, dtype=np.float64)
    if key == 'theta':
        return np.asarray(log['theta_actual'][:n] - target, dtype=np.float64)
    if key == 'psi':
        return np.array([wrap_deg(a - b) for a, b in zip(log['psi_actual'][:n], target)],   # heading needs wrap
                        dtype=np.float64)
    if key == 'phi':
        return np.asarray(log['phi_actual'][:n] - target, dtype=np.float64)
    if key == 'beta':
        return np.asarray(log['beta_actual'][:n] - target, dtype=np.float64)
    raise KeyError(key)

# Backward-compatible name used below in the notebook.
fault_deviation = operational_deviation                           # alias kept for downstream plot cells


In [ ]:
# Top-down turn trajectory with the engine-failure instant marked.
FT_PER_NM = 6076.12                                              # conversion: 1 nautical mile = 6076.12 ft

def _track_nm(log):                                              # convert NED ft track to NM relative to start
    north_nm = (np.asarray(log['x_e'], dtype=np.float64) - float(log['x_e'][0])) / FT_PER_NM
    east_nm = (np.asarray(log['y_e'], dtype=np.float64) - float(log['y_e'][0])) / FT_PER_NM
    return east_nm, north_nm

def _reference_track_nm():                                       # purely kinematic reference ground track
    north_ft = np.zeros_like(_t_grid)                            # accumulators for north / east displacement
    east_ft = np.zeros_like(_t_grid)
    for i in range(1, len(_t_grid)):                             # midpoint integrate V * (cos psi, sin psi)
        psi_mid = 0.5 * (_psi_grid[i - 1] + _psi_grid[i])
        north_ft[i] = north_ft[i - 1] + V_REF_FT_S * math.cos(float(psi_mid)) * DT
        east_ft[i] = east_ft[i - 1] + V_REF_FT_S * math.sin(float(psi_mid)) * DT
    return east_ft / FT_PER_NM, north_ft / FT_PER_NM             # return in NM

fig, ax = plt.subplots(figsize=(9, 7))                           # square-ish plot for top-down view
ref_east_nm, ref_north_nm = _reference_track_nm()
ax.plot(ref_east_nm, ref_north_nm, color='black', ls='--', lw=1.1, label='kinematic reference')

base_east_nm, base_north_nm = _track_nm(log_p4_nominal)          # healthy baseline track
ax.plot(base_east_nm, base_north_nm, color='tab:green', lw=1.2, alpha=0.8, label='healthy baseline')

for log_, color, label in [                                      # all damaged rollouts overlaid
    (log_p1, 'tab:gray', 'Phase 1 damaged'),
    (log_pid, 'tab:orange', 'PID damaged'),
    (log_p4, 'tab:blue', 'Phase 4 damaged'),
]:
    east_nm, north_nm = _track_nm(log_)
    ax.plot(east_nm, north_nm, color=color, lw=1.4, label=label)

fail_idx = int(np.argmin(np.abs(log_p4['t'] - DAMAGE_TIME)))     # find sample closest to DAMAGE_TIME on Phase 4 track
fail_east_nm, fail_north_nm = _track_nm(log_p4)
ax.scatter(fail_east_nm[fail_idx], fail_north_nm[fail_idx],      # mark engine-failure point with X
           s=95, marker='X', color='red', edgecolor='white', linewidth=0.9,
           zorder=5, label=f'engine failure, t={DAMAGE_TIME:.0f} s')
ax.annotate(f'engine failure\nt = {DAMAGE_TIME:.0f} s',          # annotate with offset arrow
            xy=(fail_east_nm[fail_idx], fail_north_nm[fail_idx]),
            xytext=(12, -28), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.0),
            color='red', fontsize=9, ha='left', va='top')

east4_nm, north4_nm = _track_nm(log_p4)                          # mark start and end of Phase 4 track
ax.scatter(east4_nm[0], north4_nm[0], s=42, marker='o', color='green',
           edgecolor='white', linewidth=0.7, zorder=4, label='start')
ax.scatter(east4_nm[-1], north4_nm[-1], s=48, marker='s', color='tab:purple',
           edgecolor='white', linewidth=0.7, zorder=4, label='end')

ax.set_aspect('equal', adjustable='box')                         # equal aspect: shape of the turn is undistorted
ax.set_xlabel('east displacement, NM')
ax.set_ylabel('north displacement, NM')
ax.set_title('Coordinated-turn ground track with mid-turn engine failure')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=9)
plt.tight_layout(); plt.show()


## 9. Nominal reference vs actual — six-panel comparison

Each panel: operational reference (dashed black), Phase 1 damaged actual (gray), Phase 4 damaged actual (blue). The vertical red line marks the engine failure at t = 50 s. The bank-angle and heading panels are the most informative — they show the aircraft executing the turn through the failure event in both configurations.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 9.5), sharex=True)   # 3x2 grid sharing x-axis

# For this paper figure, the dashed reference is the achievable healthy closed-loop
# maneuver, not the raw kinematic command. This removes nominal maneuver lag from
# the visual comparison: before DAMAGE_TIME the damaged and healthy rollouts are
# identical, so every panel overlays exactly.
plot_panels = [                                                  # one entry per panel: (log key, y label)
    ('V_actual', 'true airspeed, ft/s'),
    ('h_actual', 'altitude, ft'),
    ('theta_actual', 'pitch theta, deg'),
    ('psi_actual', 'heading psi, deg'),
    ('phi_actual', 'bank phi, deg'),
    ('beta_actual', 'sideslip beta, deg'),
]

def _plot_actual_vs_nominal(ax, key, ylabel):                    # populate one panel
    n_ref = min(len(log_p4['t']), len(log_p4_nominal['t']))      # align length with healthy baseline
    t_ref = log_p4['t'][:n_ref]
    target_key = key.replace('_actual', '')                      # 'V_actual' -> 'V', etc.
    y_ref = operational_target(log_p4, log_p4_nominal, target_key)[:n_ref]
    ax.plot(t_ref, y_ref, color='black', ls='--', lw=1.1, label='operational reference')

    for log_, color, label in [                                  # overlay every damaged rollout
        (log_p1, 'tab:gray', 'Phase 1 damaged'),
        (log_pid, 'tab:orange', 'PID damaged'),
        (log_p4, 'tab:blue', 'Phase 4 damaged'),
    ]:
        n = min(len(log_['t']), n_ref)
        ax.plot(log_['t'][:n], log_[key][:n], color=color, lw=1.2, label=label)

    ax.axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5, label='engine failure')   # damage marker
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

for ax, (key, ylabel) in zip(axes.flat, plot_panels):            # iterate panels in row-major order
    _plot_actual_vs_nominal(ax, key, ylabel)

axes[2, 0].set_xlabel('time, s')                                 # only bottom row gets x-label
axes[2, 1].set_xlabel('time, s')
axes[0, 0].legend(loc='upper right', ncol=2, fontsize=8)         # one legend on the first panel
fig.suptitle('Coordinated banked turn with mid-turn engine failure — operational reference vs actual')
plt.tight_layout(); plt.show()


## 9b. Operational deviations over time

Operational deviations `Delta(t) = damaged(t) - healthy_baseline(t)` for every channel. The horizontal zero line means the failure has introduced no deviation from the nominal maneuver. The red vertical line marks the engine failure at t = 50 s; the shaded band after it is the post-damage window where the RMS is computed.


In [ ]:
# Operational deviations: actual trajectory minus the operational reference.
# This removes nominal tracking lag from the error plot. Before DAMAGE_TIME the
# two deterministic rollouts should coincide up to numerical roundoff.
dev_panels = [                                                   # one entry per panel: (key, label, name)
    ("V",     "Delta V (ft/s)",    "speed"),
    ("h",     "Delta h (ft)",      "altitude"),
    ("theta", "Delta theta (deg)", "pitch"),
    ("psi",   "Delta psi (deg)",   "heading"),
    ("phi",   "Delta phi (deg)",   "bank"),
    ("beta",  "Delta beta (deg)",  "sideslip"),
]

def _rms(y):                                                     # numerically stable RMS helper
    y = np.asarray(y, dtype=np.float64)
    if y.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean(y ** 2)))

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)     # 3x2 deviation panels
for ax, (key, ylabel, _label) in zip(axes.flat, dev_panels):
    t1 = common_t(log_p1, log_p1_nominal)                        # aligned time vectors per controller
    tpid = common_t(log_pid, log_pid_nominal)
    t4 = common_t(log_p4, log_p4_nominal)
    d1 = fault_deviation(log_p1, log_p1_nominal, key)            # operational deviations per controller
    dpid = fault_deviation(log_pid, log_pid_nominal, key)
    d4 = fault_deviation(log_p4, log_p4_nominal, key)
    post_mask_p1 = t1 >= DAMAGE_TIME                             # post-damage mask for RMS computation
    post_mask_pid = tpid >= DAMAGE_TIME
    post_mask_p4 = t4 >= DAMAGE_TIME

    ax.plot(t1, d1, color="tab:gray", lw=1.2, label="Phase 1")    # plot all three deviation traces
    ax.plot(tpid, dpid, color="tab:orange", lw=1.2, label="PID")
    ax.plot(t4, d4, color="tab:blue", lw=1.2, label="Phase 4")
    ax.axhline(0.0, color="black", ls="--", lw=1.0, alpha=0.4)    # zero reference line
    ax.axvline(DAMAGE_TIME, color="red", ls="--", alpha=0.6, label="engine failure")
    ax.axvspan(DAMAGE_TIME, TOTAL_TIME, color="red", alpha=0.05)  # shade post-damage window

    stacked = np.concatenate([d1, dpid, d4])                      # robust y-limits via 1-99 percentiles
    lo, hi = np.percentile(stacked, [1.0, 99.0])
    pad = 0.1 * (hi - lo) if hi > lo else max(1e-4, 0.1 * abs(hi))
    ax.set_ylim(lo - pad, hi + pad)

    rms_p1 = _rms(d1[post_mask_p1])                               # post-damage RMS per controller
    rms_pid = _rms(dpid[post_mask_pid])
    rms_p4 = _rms(d4[post_mask_p4])
    pre_max_p4 = float(np.max(np.abs(d4[t4 < DAMAGE_TIME]))) if np.any(t4 < DAMAGE_TIME) else float("nan")  # pre-damage max
    ax.set_title(f"{ylabel}    P1: {rms_p1:.3f}   PID: {rms_pid:.3f}   P4: {rms_p4:.3f}   pre: {pre_max_p4:.1e}",
                 fontsize=10)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

axes[2, 0].set_xlabel("time (s)")
axes[2, 1].set_xlabel("time (s)")
axes[0, 0].legend(loc="upper right", fontsize=9)
fig.suptitle("Operational deviation: actual trajectory minus operational reference", y=1.00)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(13, 3.6))                         # cumulative |Delta psi| timeline
t1 = common_t(log_p1, log_p1_nominal)
tpid = common_t(log_pid, log_pid_nominal)
t4 = common_t(log_p4, log_p4_nominal)
abs_d_psi_p1 = np.abs(fault_deviation(log_p1, log_p1_nominal, "psi"))
abs_d_psi_pid = np.abs(fault_deviation(log_pid, log_pid_nominal, "psi"))
abs_d_psi_p4 = np.abs(fault_deviation(log_p4, log_p4_nominal, "psi"))
cum_p1 = np.cumsum(abs_d_psi_p1) * DT                             # rectangle-rule cumulative integral
cum_pid = np.cumsum(abs_d_psi_pid) * DT
cum_p4 = np.cumsum(abs_d_psi_p4) * DT
ax.plot(t1, cum_p1, color="tab:gray", lw=1.4, label="Phase 1")
ax.plot(tpid, cum_pid, color="tab:orange", lw=1.4, label="PID")
ax.plot(t4, cum_p4, color="tab:blue", lw=1.4, label="Phase 4")
ax.axvline(DAMAGE_TIME, color="red", ls="--", alpha=0.6, label="engine failure")
ax.axvspan(DAMAGE_TIME, TOTAL_TIME, color="red", alpha=0.05)
ax.set_xlabel("time (s)")
ax.set_ylabel(r"$\int_0^t |\Delta\psi(\tau)|\, d\tau$, deg$\cdot$s")
ax.set_title("Cumulative absolute operational heading deviation")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


## 10. Control surfaces and throttle

Side-by-side commands. Phase 4's L4 outer adds tiny residual references on top of the envelope allocator's command, so the two traces should largely overlap with localized differences when L4 / macro-actions are most active.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6.5), sharex=True)   # 2x2 panels for de, da, dr, throttle
ctrl_panels = [
    ('de', 'elevator de, deg'),
    ('da', 'aileron da, deg'),
    ('dr', 'rudder dr, deg'),
    ('throttle', 'throttle [0,1]'),
]
for ax, (key, ylabel) in zip(axes.flat, ctrl_panels):            # iterate panels
    ax.plot(log_p1['t'], log_p1[key], color='tab:gray', lw=1.0, label='Phase 1')
    ax.plot(log_pid['t'], log_pid[key], color='tab:orange', lw=1.0, label='PID')
    ax.plot(log_p4['t'], log_p4[key], color='tab:blue', lw=1.0, label='Phase 4')
    ax.axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.5)
    ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3)
axes[1,0].set_xlabel('time, s'); axes[1,1].set_xlabel('time, s')
axes[0,0].legend(loc='upper right', fontsize=9)
fig.suptitle('Control surfaces & throttle — PID baseline vs UFTC')
plt.tight_layout(); plt.show()


## 11. Composite Lyapunov monitor — V_total and alarm timeline (Phase 4)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True,    # top: V_total, bottom: alarm timeline
                         gridspec_kw={'height_ratios': [3, 1]})
mu_pred = float(ctl_p4.monitor.mu_uub_pred)                       # predicted UUB radius from the monitor
warn_th = ctl_p4.cfg.monitor_alarm_warn_frac * mu_pred             # WARN absolute threshold
crit_th = ctl_p4.cfg.monitor_alarm_critical_frac * mu_pred         # CRITICAL absolute threshold
axes[0].plot(log_p4['t'], log_p4['V_total'], color='tab:blue', lw=1.3, label=r'$V_{\mathrm{total}}(t)$')
axes[0].axhline(mu_pred, color='red', lw=1.3, label=fr'$\mu_{{UUB}}={mu_pred:.3f}$')
axes[0].axhline(crit_th, color='red', lw=1.0, ls='--',
                label=f'CRITICAL = {ctl_p4.cfg.monitor_alarm_critical_frac:.2f} mu')
axes[0].axhline(warn_th, color='darkorange', lw=1.0, ls='--',
                label=f'WARN = {ctl_p4.cfg.monitor_alarm_warn_frac:.2f} mu')
axes[0].axvline(DAMAGE_TIME, color='red', ls=':', alpha=0.55, label='engine failure')
axes[0].set_ylabel(r'$V_{\mathrm{total}}$'); axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper left', ncol=2)
axes[1].step(log_p4['t'], log_p4['alarm_level'], color='tab:red', where='post', lw=1.4)  # discrete alarm level
axes[1].axvline(DAMAGE_TIME, color='red', ls=':', alpha=0.55)
axes[1].set_yticks([0,1,2]); axes[1].set_yticklabels(['OK','WARN','CRITICAL'])
axes[1].set_ylim(-0.2, 2.4)
axes[1].set_xlabel('time, s'); axes[1].set_ylabel('alarm')
axes[1].grid(True, alpha=0.3)
axes[0].set_title('Phase 4 monitor — V_total and alarm during banked turn + engine failure')
plt.tight_layout(); plt.show()

alarm_int = log_p4['alarm_level']                                 # numeric alarm series
ok_frac   = float(np.mean(alarm_int == 0))                        # fraction of time at each alarm level
warn_frac = float(np.mean(alarm_int == 1))
crit_frac = float(np.mean(alarm_int == 2))
print(f'V_total range: [{float(np.min(log_p4["V_total"])):.3f}, {float(np.max(log_p4["V_total"])):.3f}], '
      f'mean = {float(np.mean(log_p4["V_total"])):.3f}')
print(f'Alarm time fractions OK / WARN / CRITICAL: '
      f'{ok_frac*100:.1f}% / {warn_frac*100:.1f}% / {crit_frac*100:.1f}%')
print(f'Alarm transitions: {len(log_p4["alarm_transitions"])}')
for step_idx, t_, prev, new in log_p4['alarm_transitions'][:20]:  # show first 20 transitions
    print(f'  step {step_idx:>5d}, t={t_:>6.2f} s : {prev:>9s} -> {new}')
macro_kinds = Counter(k for _, _, k, _ in log_p4['macro_events'])  # tally fired macro-actions
print(f'Macro-actions fired (total {len(log_p4["macro_events"])}):')
for k_, v_ in sorted(macro_kinds.items(), key=lambda kv: -kv[1]):
    print(f'  {k_:32s} : {v_} fires')


## 12. Per-engine thrust


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.2))                         # single panel for all four engines + total
labels = {'T1': 'engine 1: left outer (failed)',
          'T2': 'engine 2: left inner',
          'T3': 'engine 3: right inner',
          'T4': 'engine 4: right outer'}
for k_, lab in labels.items():                                    # plot each engine's thrust trace
    ax.plot(log_p4['t'], log_p4[k_], label=lab)
ax.plot(log_p4['t'], log_p4['T_total'], color='black', ls='--', lw=1.2, label='total thrust')   # total dashed
ax.axvline(DAMAGE_TIME, color='red', ls='--', alpha=0.45, label='engine failure')
ax.set_xlabel('time, s'); ax.set_ylabel('thrust, lb')
ax.set_title('Per-engine thrust during turn (Phase 4)')
ax.grid(True, alpha=0.3); ax.legend(loc='upper right', ncol=2)
plt.tight_layout(); plt.show()


## 13. Fault-induced RMS comparison

Quantitative summary of **actual minus operational-reference deviation** in the post-damage window `t ∈ [50, 300] s`. We report RMS of the operational deviation per channel for both configurations.


In [ ]:
post_t4 = common_t(log_p4, log_p4_nominal) >= DAMAGE_TIME         # post-damage masks per controller
post_t1 = common_t(log_p1, log_p1_nominal) >= DAMAGE_TIME
post_tpid = common_t(log_pid, log_pid_nominal) >= DAMAGE_TIME
pre_t4 = common_t(log_p4, log_p4_nominal) < DAMAGE_TIME           # pre-damage masks (sanity check)
pre_t1 = common_t(log_p1, log_p1_nominal) < DAMAGE_TIME
pre_tpid = common_t(log_pid, log_pid_nominal) < DAMAGE_TIME

def rms(y):                                                       # local RMS helper (reuse pattern)
    return float(np.sqrt(np.mean(np.asarray(y, dtype=np.float64) ** 2)))

rows = []                                                         # rows for the post-damage RMS table
pre_rows = []                                                     # rows for pre-damage max table
for key, label, unit in [
    ('V', 'speed deviation', 'ft/s'),
    ('h', 'altitude deviation', 'ft'),
    ('theta', 'pitch deviation', 'deg'),
    ('psi', 'heading deviation', 'deg'),
    ('phi', 'bank deviation', 'deg'),
    ('beta', 'sideslip deviation', 'deg'),
]:
    d1 = fault_deviation(log_p1, log_p1_nominal, key)             # operational deviations per controller
    dpid = fault_deviation(log_pid, log_pid_nominal, key)
    d4 = fault_deviation(log_p4, log_p4_nominal, key)
    r1 = rms(d1[post_t1])                                         # post-damage RMS per controller
    rpid = rms(dpid[post_tpid])
    r4 = rms(d4[post_t4])
    pre1 = float(np.max(np.abs(d1[pre_t1]))) if np.any(pre_t1) else float('nan')      # pre-damage |max|
    prepid = float(np.max(np.abs(dpid[pre_tpid]))) if np.any(pre_tpid) else float('nan')
    pre4 = float(np.max(np.abs(d4[pre_t4]))) if np.any(pre_t4) else float('nan')
    rows.append((label, unit, r1, rpid, r4))
    pre_rows.append((label, unit, pre1, prepid, pre4))

print(f'Operational RMSE (actual - operational reference), t ∈ [{DAMAGE_TIME:.0f}, {TOTAL_TIME:.0f}] s')
print(f'{"channel":<22s}  {"unit":<6s}  {"Phase 1":>10s}  {"PID":>10s}  {"Phase 4":>10s}  {"P4/PID":>8s}')
print('-' * 79)
for label, unit, r1, rpid, r4 in rows:                            # print one row per channel
    ratio = r4 / rpid if rpid > 0 else float('nan')
    print(f'{label:<22s}  {unit:<6s}  {r1:>10.4f}  {rpid:>10.4f}  {r4:>10.4f}  {ratio:>8.3f}')

print()
print(f'Pre-failure max |actual - operational reference|, t < {DAMAGE_TIME:.0f} s')
print(f'{"channel":<22s}  {"unit":<6s}  {"Phase 1":>10s}  {"PID":>10s}  {"Phase 4":>10s}')
print('-' * 68)
for label, unit, pre1, prepid, pre4 in pre_rows:                  # print pre-failure determinism check
    print(f'{label:<22s}  {unit:<6s}  {pre1:>10.3e}  {prepid:>10.3e}  {pre4:>10.3e}')

# Final-state summary
print()
print('Final operational deviation (last logged sample):')
for tag, log_, base_ in [
    ('Phase 1', log_p1, log_p1_nominal),
    ('PID', log_pid, log_pid_nominal),
    ('Phase 4', log_p4, log_p4_nominal),
]:
    n = _aligned_len(log_, base_)
    print(f'  {tag}:  Delta V={fault_deviation(log_, base_, "V")[n-1]:+8.3f} ft/s, '
          f'Delta h={fault_deviation(log_, base_, "h")[n-1]:+8.2f} ft, '
          f'Delta phi={fault_deviation(log_, base_, "phi")[n-1]:+7.3f} deg, '
          f'Delta psi={fault_deviation(log_, base_, "psi")[n-1]:+7.3f} deg')


## 14. 3D WebGL viewer — open-loop vs PID vs Phase 4

Build three self-contained B-747 WebGL viewers from the same maneuver: true open-loop, classical PID baseline, and the full Phase 4 UFTC stack. Use the generated links to open the full interactive viewer in a separate browser tab.


In [ ]:
import json                                                     # JSON for serialising the flight log
from IPython.display import HTML, display                         # for inline iframe display in the notebook
from tensoraerospace.visualization.three_d import build_html      # builds a self-contained WebGL HTML page

FT_TO_M = 0.3048                                                  # conversion: 1 ft -> 0.3048 m
UFTC_3D_DIR = Path.cwd() if Path.cwd().name == 'uftc' else Path('example/reinforcement_learning/uftc')   # output dir

def _pad_or_trim(values, n):                                      # ensure a series has exactly n samples
    arr = np.asarray(values, dtype=np.float64).reshape(-1)
    if arr.size < n:
        arr = np.concatenate([arr, np.full(n - arr.size, arr[-1] if arr.size else 0.0)])  # pad with last value
    elif arr.size > n:
        arr = arr[:n]                                             # trim to length
    return arr

def build_uftc_b747_3d_flight_log(log, *, damage_time=DAMAGE_TIME):  # build the JSON-ready dict consumed by build_html
    t = np.asarray(log['t'], dtype=np.float64)
    n = len(t)
    x_e_m = _pad_or_trim(log['x_e'], n) * FT_TO_M                 # NED north in metres
    y_e_m = _pad_or_trim(log['y_e'], n) * FT_TO_M                 # NED east in metres
    h_m = _pad_or_trim(log['h_actual'], n) * FT_TO_M              # altitude in metres
    attitude_rad = np.column_stack([                              # attitude as (roll, pitch, yaw) in rad
        np.radians(_pad_or_trim(log['phi_actual'], n)),
        np.radians(_pad_or_trim(log['theta_actual'], n)),
        np.radians(_pad_or_trim(log['psi_actual'], n)),
    ])
    params = default_parameters(B747Configuration.NOMINAL)        # mass / inertia for the metadata block
    healthy_state = {                                             # all-healthy damage snapshot
        'mu': {'elevator': 1.0, 'aileron': 1.0, 'rudder': 1.0, 'throttle': 1.0},
        'jam': {'elevator': None, 'aileron': None, 'rudder': None, 'throttle': None},
        'tau': {'elevator': 0.0, 'aileron': 0.0, 'rudder': 0.0, 'throttle': 0.0},
        'mu_floor': {'elevator': 0.0, 'aileron': 0.0, 'rudder': 0.0, 'throttle': 0.0},
        'engines_mu': {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0},
        'flap_jam_config': None,
    }
    failed_state = json.loads(json.dumps(healthy_state))          # deep-copy then break engine #1
    failed_state['engines_mu']['1'] = 0.0

    return {                                                      # final dict consumed by build_html
        'version': 1,
        'metadata': {
            'model': 'B-747',
            'aircraft_type': 'b747',
            'dt': float(DT),
            'n_steps': int(n),
            'airspeed': float(_pad_or_trim(log['V_actual'], n)[0] * FT_TO_M),    # initial airspeed in m/s
            'split_stab': False,
            'params': {
                'weight_lb': float(params.weight_lb),
                'S_ft2': float(params.S_ft2),
                'b_ft': float(params.b_ft),
                'cbar_ft': float(params.cbar_ft),
                'Ix': float(params.Ix),
                'Iy': float(params.Iy),
                'Iz': float(params.Iz),
                'Ixz': float(params.Ixz),
            },
        },
        'geometry': {'aircraft_type': 'b747', 'sections': []},   # geometry placeholder (viewer fills via GLB)
        'trajectory': {                                          # all per-step time series in SI / radians
            'time': t.tolist(),
            'position': np.column_stack([x_e_m, y_e_m, -h_m]).tolist(),    # (north, east, down) in metres
            'attitude': attitude_rad.tolist(),
            'alpha': np.full(n, float(trim_result.alpha_rad)).tolist(),    # constant alpha approximation
            'beta': np.radians(_pad_or_trim(log['beta_actual'], n)).tolist(),
            'wx': np.radians(_pad_or_trim(log['p'], n)).tolist(),
            'wy': np.radians(_pad_or_trim(log['q'], n)).tolist(),
            'wz': np.radians(_pad_or_trim(log['r'], n)).tolist(),
            'stab': np.radians(_pad_or_trim(log['de'], n)).tolist(),
            'ail': np.radians(_pad_or_trim(log['da'], n)).tolist(),
            'dir': np.radians(_pad_or_trim(log['dr'], n)).tolist(),
            'throttle': _pad_or_trim(log['throttle'], n).tolist(),
            'altitude_m': h_m.tolist(),
            'airspeed_mps': (_pad_or_trim(log['V_actual'], n) * FT_TO_M).tolist(),
            'references': {                                       # reference series for chart overlays
                'V': (_pad_or_trim(log['V_ref'], n) * FT_TO_M).tolist(),
                'h': (_pad_or_trim(log['h_ref'], n) * FT_TO_M).tolist(),
                'theta': _pad_or_trim(log['theta_ref'], n).tolist(),
                'roll': _pad_or_trim(log['phi_ref'], n).tolist(),
                'yaw': _pad_or_trim(log['psi_ref'], n).tolist(),
                'beta': _pad_or_trim(log['beta_ref'], n).tolist(),
            },
        },
        'damage_events': [                                        # damage timeline, displayed in the viewer
            {
                'time': float(damage_time),
                'label': 'left_outer_engine_flameout_mid_turn',
                'event_type': 'EngineFailureEvent',
                'kind': 'EngineFailureEvent',
                'payload': {'engine_id': 1, 'thrust_fraction': 0.0},
            }
        ],
        'damage_state_history': [                                 # piecewise-constant damage state
            {'time': 0.0, 'state': healthy_state},
            {'time': float(damage_time), 'state': failed_state},
        ],
    }

# Build comparable viewers: true open-loop, PID baseline, and Phase 4 UFTC.
UFTC_3D_DIR.mkdir(parents=True, exist_ok=True)                    # ensure output dir exists
viewers = [                                                       # one viewer per controller config
    ('open_loop', log_open_loop, 'B-747 — true open-loop (no feedback)'),
    ('pid',       log_pid, 'B-747 — PID baseline'),
    ('phase4',    log_p4, 'B-747 — Phase 4 UFTC'),
]
viewer_paths = {}                                                 # paths of generated HTML files
for tag, log, title in viewers:
    path = UFTC_3D_DIR / f'uftc_b747_coordinated_turn_engine_failure_3d_{tag}.html'
    flight_log = build_uftc_b747_3d_flight_log(log)               # build the JSON-ready dict
    path.write_text(build_html(flight_log, title=title), encoding='utf-8')   # render full HTML page
    viewer_paths[tag] = path
    print(f'3D viewer saved: {path.resolve()}')

# Responsive iframes for in-notebook comparison; each link opens the full viewer.
display(HTML(
    '<div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(360px, 1fr)); gap:10px;">'
    + ''.join(
        f'<div>'
        f'<div style="font-weight:600; margin-bottom:4px;">{title}</div>'
        f'<a href="{viewer_paths[tag].name}" target="_blank">Open in new tab</a>'
        f'<iframe src="{viewer_paths[tag].name}" width="100%" height="620" '
        f'style="border:0; border-radius:6px; background:#111;"></iframe>'
        f'</div>'
        for tag, _, title in viewers
    )
    + '</div>'
))


## 15. Notes & honest caveats

* **Reference injection point.** UFTC's `reference` argument is held at zero. The maneuver is injected through the *envelope-allocator setpoint* `ref_t` consumed by `uftc_state_transform_dyn` and `state_feedback_action_dyn`. UFTC sees a non-trivial drifting error and the cascade compensates online. This is the cleaner path because (a) it preserves the controller's plant-agnostic interface (no controller-side change to support a maneuver), and (b) the existing engine-out trim blend (`trim_action_for_loss`) lives in the envelope allocator anyway.
* **PID comparison baseline.** The PID rollout uses one fixed gain set and the fixed healthy trim point for the whole episode. It does not receive the engine-loss estimate, does not switch to one-engine-out trim, and does not change targets after the failure. Its role is a classical-control baseline for the fault transient.
* **Damage trigger.** A local `DamageProfile` is built with `EngineFailureEvent(trigger_time=50.0, engine_id=1, thrust_fraction=0.0)`, mirroring the stock `LEFT_OUTER_ENGINE_FAILURE` preset but mid-turn rather than at t=10s.
* **Pre-trained L4 actor mismatch.** The actor in `artifacts/dsac/b747_engine_out_v1/` was trained on the stationary-trim engine-out scenario, *not* on a banked-turn schedule. We expect Phase 4 to be roughly comparable to Phase 1 — possibly slightly worse on yaw because the L4 reference correction is mis-aligned with the maneuver. A curriculum across maneuvers (level cruise + various bank schedules + engine-out at random t) would close that gap; the current demo shows the framework runs end-to-end on a non-stationary task, not that L4 is optimal here.
* **Determinism.** `np.random.seed(0)`, `torch.manual_seed(0)`, `AAINDIConfig(seed=0)`, `l4_seed=0`. The same rollout reproduces bit-for-bit across runs.
* **Coordinated-turn approximation.** Bank φ is set directly via a piecewise-linear schedule; ψ_ref(t) is the trapezoidal integral of the coordinated-turn rate `g·tan(φ)/V`. The aircraft is not constrained to fly a *true* coordinated turn — the controller adapts to whatever (φ_ref, ψ_ref) trajectory we feed it.
